# Auto-fusion over the whole corpus

Auto-fusion is the shipped incumbent this project is trying to replace: `POST /v1/classify`
returns one integer 0–9 per query, and the production bands turn it into a route
(0–2 dense, 3–6 rrf, 7–9 sparse). It never sees the corpus — it reads the query text alone.

This notebook rehearses the classify call on a handful of queries, then reports on
`src/data/route_labels/autofusion_cache.parquet` once the script below has filled it.

| mode | what it does | cost |
| --- | --- | --- |
| **A** | one query end to end: golden scores, the wire payload, the response | 1 call |
| **B** | 10 queries weighted toward the degenerate shapes, plus the measured call rate | 10 calls |
| **C** | all 46,142 rows — `src/scripts/autofusion_fill.py`, not a cell | ~46k calls, ~$2.18 |
| **D** | stats over the finished cache | free, no network |

Run mode B first and read the rate it measures — that number, not the dollar figure, decides
whether the full run is an afternoon or two days. Mode C itself left the notebook: ~46k calls
over hours needs a process that survives a closed laptop lid, so it lives in the script and
this notebook only reads its coverage back.

In [41]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Config

`SEED` fixes which queries modes A and B draw. `QUERY_WIDTH` is how much of a query the
tables print — pandas defaults to 50 characters, which cuts every prose query mid-sentence.
`None` uncaps it, but the corpus runs to 21,111 characters and the `clerc` / `freshstack`
rows are whole Stack Overflow posts, so uncapped is less readable rather than more. To read
one row in full, print it: `print(examples.loc[43, "query"])`.

The full run's knobs (`--chunk`, `--limit`) belong to the script, not here — `route_batch`
saves the whole cache once per call, so chunking is what bounds how much a killed run
replays.

In [ ]:
import json
import time

import pandas as pd
from IPython.display import display

from hybrid_search_rrf_dataset.fusion import StrategyName
from hybrid_search_rrf_dataset.objective import RouterObjective
from hybrid_search_rrf_dataset.router import (
    LABELS_PATH,
    SCORE_COLS,
    AutoFusionRouter,
    Representation,
    RouterExperiment,
    _ROUTE_ORDER,
    _production_route,
    _score_matrix,
    _six_column,
    decisive_rows,
)
from scripts.autofusion_fill import GuardedScoreClient, _remaining

SAMPLE = {"all_tied": 7, "all_zero": 1, "routes_differ": 2}
SEED = 0
QUERY_WIDTH = 200

pd.set_option("display.max_colwidth", QUERY_WIDTH)

OBJ = RouterObjective()
GOLD_COLS = [SCORE_COLS[s] for s in _ROUTE_ORDER]


def read_cache() -> pd.DataFrame:
    """Persisted classify scores, renamed clear of `labels.score`."""
    return pd.read_parquet(AutoFusionRouter.CACHE_PATH).rename(columns={"score": "af_score"})


def decompose(score):
    """The objective's two terms, read back out of the composite: (hit@1, NDCG@k)."""
    hit = score >= OBJ.hit_weight
    return hit, (score - hit * OBJ.hit_weight) / OBJ.ndcg_weight


labels = pd.read_parquet(LABELS_PATH)
assert (labels["metric_name"] == OBJ.name).all(), "decompose() assumes the labelled objective"
print(f"{len(labels):,} queries")
print(labels["shape"].value_counts().to_string())

## The client

`LLMScoreClient` has no retry and no range check, and the cache is write-once — a value
that lands in it is never fetched again. The endpoint returned `upstream_timeout` on the
first live probe, and `_production_route` bands *any* integer without complaint, so a
score of 47 would be silently served as sparse for the rest of the project.

Both gaps close in `GuardedScoreClient` rather than in `router.py`, because
`AutoFusionRouter` already takes an injected client. It lives in the fill script, imported
above: the 46k-call run is the one that cannot afford to be missing a retry, so the guard
belongs where that run can see it — and one definition means the notebook and the script
cannot drift.

## Mode A — one query, end to end

Drawn from the decisive rows, so there is a real right answer to compare against. The
payload is printed as the literal JSON that goes on the wire.

The golden scores are split back into the objective's two terms. `hit_weight > ndcg_weight`
keeps the ranges disjoint, so `hit@1` and `NDCG@k` are both recoverable from the composite —
nothing is recomputed. Read them together: `1.0` against `0.15` is one route winning the
top-1 slot while the other still holds the answer at rank 3, not a route that returned
nothing.

In [43]:
row = decisive_rows(labels).sample(1, random_state=SEED).iloc[0]

print("query:  ", row["query"])
gold = pd.DataFrame(
    {"score": row[GOLD_COLS].astype(float).to_numpy()},
    index=[r.value for r in _ROUTE_ORDER],
)
gold["hit@1"], gold["ndcg@10"] = decompose(gold["score"])
display(gold[["hit@1", "ndcg@10", "score"]].round(3))
print("winner: ", row["winner"])

payload = {"text": str(row["query"])[:4096]}
print("payload:", json.dumps(payload))

score = GuardedScoreClient().score(str(row["query"]))
print(f"score:   {score}  ->  {_production_route(score)}")

query:   what are fossilized bones made of?


,hit@1,ndcg@10,score
dense_only,False,0.500,0.150
pure_rrf,False,0.631,0.189
sparse_only,True,1.000,1.000


winner:  sparse_only
payload: {"text": "what are fossilized bones made of?"}
score:   1  ->  dense_only


## Mode B — 10 queries, weighted toward the outliers

`all_tied` and `all_zero` rows have no winning route at all — the three scores are equal, or
all zero. Auto-fusion never sees that; it answers from the query text either way. What it
emits on those rows is the thing to look at here.

This goes through `AutoFusionRouter`, so the 10 calls land in the real cache and rehearse
exactly what mode C does. The measured rate is only meaningful on a cold cache — on a
re-run the calls are served from disk and the number collapses to ~0.

In [70]:
sample = pd.concat(
    [labels[labels["shape"] == shape].sample(n, random_state=SEED) for shape, n in SAMPLE.items()]
)

pd.set_option("display.max_colwidth", 300)

router = AutoFusionRouter(client=GuardedScoreClient())
start = time.perf_counter()
router.route_batch(sample, desc="demo")
rate = (time.perf_counter() - start) / len(sample)

demo = sample.merge(read_cache(), on=["dataset", "query_id"])
demo["route"] = demo["af_score"].map(_production_route)
display(demo[["query", "shape", *GOLD_COLS, "af_score", "route"]])

print(f"{rate:.2f}s/call  ->  {len(labels) * rate / 3600:.1f}h for all {len(labels):,}")

,query,shape,score_dense_only,score_pure_rrf,score_sparse_only,af_score,route
0,Is it legal to unlock my iPhone myself?,all_tied,1.000000,1.0,1.0,1,dense_only
1,kindred hospital north indiana phone number,all_tied,1.000000,1.0,1.0,3,pure_rrf
2,How do the data collection methods for ensemble simulation in E3SM LS4P Task3-CTRL monthly data compare to those used in the CMIP6 model runs?,all_tied,1.000000,1.0,1.0,4,pure_rrf
3,"S & P 500 index. What plaintiffs describe is a species of securities manipulation called “fraud-on-the-market.” The fraud on the market theory is based on the hypothesis that, in an open and developed securities market, the price of a company’s stock is determined by the available material infor...",all_tied,1.000000,1.0,1.0,3,pure_rrf
4,reciprocate meaning,all_tied,1.000000,1.0,1.0,1,dense_only
5,is lvl stronger than lsl?,all_tied,1.000000,1.0,1.0,3,pure_rrf
6,What is a lone worker safety policy?,all_tied,1.000000,1.0,1.0,1,dense_only
7,Flora of Bolivia that is also in the Leeward Islands,all_zero,0.000000,0.0,0.0,3,pure_rrf
8,"""result.""""). Instead, it tracks the ordinary understanding of the term, as discussed above and as reflected in the way the Supreme Court has used the term. The Court's decision in California is a good example. Recall that the Court in that case affirmed that the federal government """"controlled""""...",routes_differ,0.129203,1.0,1.0,3,pure_rrf
9,"""and are recognized by consumers. The reason for this principle is simple. Evidence that other sellers in the same industry as the plaintiff extensively use marks that are similar to the plaintiff's mark shows that """"consumers may not 'associate [the plaintiff's] mark with a unique source,' """" a...",routes_differ,0.000000,1.0,1.0,5,pure_rrf


0.00s/call  ->  0.0h for all 46,142


## Mode C — the full run, from a terminal

```bash
poetry run python src/scripts/autofusion_fill.py --plan          # coverage, no spend
poetry run python src/scripts/autofusion_fill.py --limit 10      # measure the call rate
poetry run python src/scripts/autofusion_fill.py 2>&1 | tee autofusion_fill.log
```

Idempotent: the script skips `(dataset, query_id)` pairs already in the cache, so a Ctrl-C
or a dead laptop costs at most `--chunk` queries and never re-spends on a scored query.
Nothing about it needs this kernel — the cell below only asks it for coverage.

One caveat: `to_parquet` is not atomic. If the process dies mid-save the cache file can be
truncated, in which case the next `--plan` fails on the read rather than silently starting
over; copy the cache aside before resuming a run that crashed hard.

In [45]:
todo = _remaining(labels)
print(f"cached  {len(labels) - len(todo):,} / {len(labels):,}")
print(f"todo    {len(todo):,}")

cached  46,127 / 46,142
todo    15


## Mode D — stats

Reads the cache only. Safe to re-run and iterate on while mode C is still going; it just
reports on whatever has been scored so far.

In [46]:
scored = labels.merge(read_cache(), on=["dataset", "query_id"])
scored["route"] = scored["af_score"].map(_production_route)
scored["sparse_minus_dense"] = (
    scored[SCORE_COLS[StrategyName.SPARSE_ONLY]] - scored[SCORE_COLS[StrategyName.DENSE_ONLY]]
)
print(f"{len(scored):,} of {len(labels):,} queries scored")

46,127 of 46,142 queries scored


### 1 — What the scale emits, and whether it tracks anything

Each digit's share of the corpus, the route it bands into, and the mean
`sparse_only − dense_only` score gap of the queries that got it. The classifier claims
higher digits mean sparse retrieval wins; if that is true, the last column rises with the
digit.

In [47]:
display(
    scored.groupby("af_score").agg(
        n=("query_id", "size"),
        route=("route", "first"),
        sparse_minus_dense=("sparse_minus_dense", "mean"),
    ).round(4)
)

,n,route,sparse_minus_dense
af_score,,,
0,317,dense_only,-0.0929
1,16337,dense_only,-0.1627
2,35,dense_only,-0.1501
3,25276,pure_rrf,-0.0644
4,390,pure_rrf,-0.0056
5,2900,pure_rrf,0.1657
6,11,pure_rrf,0.5891
7,457,sparse_only,0.2430
8,392,sparse_only,0.0859


### 2 — Behaviour on the degenerate shapes

Mode B's question at corpus scale. `all_tied` and `all_zero` rows carry no signal about
which route to serve.

In [48]:
display(
    scored.pivot_table(
        index="shape", columns="af_score", values="query_id", aggfunc="size", fill_value=0
    )
)

af_score,0,1,2,3,4,5,6,7,8,9
shape,,,,,,,,,,
all_tied,93,5552,15,8282,53,856,4,118,45,0
all_zero,40,1861,3,5750,89,286,0,24,120,4
routes_differ,184,8924,17,11244,248,1758,7,315,227,8


### 3 — Agreement with the labelled winner

Decisive rows only — the ones where a route beat the runner-up by the objective's decisive
margin, so there is an unambiguous correct answer.

In [49]:
dec = decisive_rows(scored)
print(f"{len(dec):,} decisive rows, agreement {(dec['route'] == dec['winner']).mean():.1%}")
display(pd.crosstab(dec["route"], dec["winner"], margins=True))

5,095 decisive rows, agreement 35.2%


winner,dense_only,pure_rrf,sparse_only,All
route,,,,
dense_only,1558,83,505,2146
pure_rrf,1561,123,1112,2796
sparse_only,29,13,111,153
All,3148,219,1728,5095


### 4 — Six-column headroom table

The same instrument `RouterExperiment` reports, so these numbers sit alongside the router's.

**This is not C1b.** C1b is (router − auto-fusion) over held-out *answerable* rows on the
grouped split with a clustered bootstrap CI against Δmin. This is auto-fusion alone over
every decisive row, with no CI — a descriptive table, not a gate reading.

In [50]:
display(pd.Series(_six_column(dec, list(dec["route"]))).to_frame("auto_fusion").T.round(4))

,const_dense_only,const_pure_rrf,const_sparse_only,oracle,router,headroom_captured
auto_fusion,0.6295,0.2084,0.3725,0.976,0.4452,-0.532


### 5 — Where it holds and where it collapses

Per-lane agreement. A classifier that reads query surface should do well on lanes with
identifier-shaped queries and badly on prose lanes.

In [51]:
display(
    dec.assign(hit=dec["route"] == dec["winner"])
    .groupby("dataset")["hit"]
    .agg(["mean", "size"])
    .sort_values("mean")
    .round(3)
)

,mean,size
dataset,,
bright-stackoverflow,0.000,6
limit,0.000,133
crumb-clinical-trial,0.000,5
freshstack-yolo,0.000,7
bright-leetcode,0.000,4
crumb-paper-retrieval,0.000,2
freshstack-laravel,0.038,26
freshstack-angular,0.056,18
crumb-code-retrieval,0.073,41


### 6 — The disagreement slice

Decisive rows where auto-fusion serves a route that contradicts the labelled winner. This is
the raw material Gate 3(a) is built from, and where the hand-ruled confusing-query
candidates get harvested.

A random sample, ordered by severity within itself. Sorting the whole slice by severity fills
the first page with `limit` — a stress set where dense fails by construction, so it holds most
of the `served_ndcg == 0` tail and says more about that lane's share than about the classifier.
Only one route can hit@1 in a decisive row (two routes that both hit are within `ndcg_weight`
of each other, under the decisive margin), so `hit@1` is False across the whole slice and
`served_ndcg` alone measures severity: `0` means the served route missed the gold entirely,
`0.631` means it sat at rank 2.

In [69]:
dis = dec[dec["route"] != dec["winner"]].copy()
served = _score_matrix(dis)[range(len(dis)), [_ROUTE_ORDER.index(r) for r in dis["route"]]]
dis["served_ndcg"] = decompose(served)[1]
blind = dis["served_ndcg"] == 0

print(f"{len(dis):,} of {len(dec):,} decisive rows contradicted ({len(dis) / max(len(dec), 1):.1%})")
print(
    f"{blind.sum():,} served a route that missed the gold entirely; "
    f"{(~blind).sum():,} had it below rank 1"
)
display(
    dis.sample(min(20, len(dis)), random_state=SEED)
    .sort_values("served_ndcg")[
        ["dataset", "query", "af_score", "route", "winner", "served_ndcg"]
    ]
    .round(3)
)

3,303 of 5,095 decisive rows contradicted (64.8%)
296 served a route that missed the gold entirely; 3,007 had it below rank 1


,dataset,query,af_score,route,winner,served_ndcg
10392,rarb-math,"Problem: Four diagonals of a regular octagon with side length 2 intersect as shown. Find the area of the shaded region. [asy]\npair A, B, C, D, E, F, G, H;\nreal x = 22.5;\npair A = dir(x);\npair B = dir(45+x);\npair C = dir(45*2+x);\npair D = dir(45*3+x);\npair E = dir(45*4+x);\npair F = dir(4...",1,dense_only,sparse_only,0.000
10842,limit,Who likes Swimsuits?,1,dense_only,sparse_only,0.000
10841,limit,Who likes Grease Ants?,1,dense_only,sparse_only,0.000
10875,limit,Who likes Popcorn?,1,dense_only,sparse_only,0.000
14228,quest,Communism books that are not about Asia,3,pure_rrf,sparse_only,0.464
10464,rarb-math,"Problem: Some people got on a bus at the terminal. At the first bus stop, 5 more people got in. Then at the second bus stop, 7 people got down and 8 more got in. If there were a total of 20 people heading to the third stop, how many people got on the bus at the terminal?",1,dense_only,sparse_only,0.500
46023,msmarco-passage-dev,the musica man cast,3,pure_rrf,dense_only,0.631
22167,gooaq,is omio a trusted website?,3,pure_rrf,dense_only,0.631
10384,rarb-math,"Problem: In right $\triangle ABC$, shown here, $AB = 15 \text{ units}$, $AC = 24 \text{ units}$ and points $D,$ $E,$ and $F$ are the midpoints of $\overline{AC}, \overline{AB}$ and $\overline{BC}$, respectively. In square units, what is the area of $\triangle DEF$?\n\n[asy]\nsize(150);\nimport o...",3,pure_rrf,sparse_only,0.631
30704,webfaq-eng,Is the Devdan Show suitable for adults without children?,3,pure_rrf,dense_only,0.631


### 7 — Auto-fusion against the router

The head-to-head. `RouterExperiment.run(autofusion=True)` already implements it (SPEC d47):
it fits `StrategyRouter` on the train split, tunes thresholds on a held-out slice of train,
then scores **both** the router and auto-fusion on the *same* held-out decisive rows with
the same six-column table. Auto-fusion needs no fit, so the split costs it nothing — it is
there to make the two rows comparable, not to train anything.

Read `representation` as the policy column: `Representation.ENGINEERED` is the router,
`auto_fusion` is the incumbent. `all_rows` is the router's *training* substrate only; both
policies are always evaluated on held-out decisive rows.

Both protocols run (SPEC d46f). `random_within_lane` is "new queries on a known
collection"; `holdout_lane` is "a collection never seen" — the router trains without
`rarb-math` and is tested only on it. The gap between the two is the router's
corpus-dependence, and auto-fusion, being corpus-blind, should barely move between them.

Two practical notes. The cell makes **no network calls** once the cache is complete, but
`AutoFusionRouter` builds its client in `__init__`, so `QDRANT_LLM_FUSION_URL`/`_KEY` still
have to be in the environment. And the assert is the money guard: any unscored query in a
held-out decisive frame would be classified live mid-comparison.

**One caveat this notebook has already measured.** The evaluation substrate is held-out
*decisive* rows, and the decisive margin deletes most of rrf's wins — rrf takes 21.9% of
`routes_differ` rows but only 4.9% of decisive ones, because a fusion route rarely beats
both of its own inputs by `hit_weight − ndcg_weight`. Auto-fusion emits rrf for ~55% of the
corpus, so this substrate reads it low. Section 4's `const_pure_rrf` column is the tell: it
collapses here while being the strongest constant on the full labelled set. A
`routes_differ` readout would need a substrate argument on `RouterExperiment`, which does
not exist yet.

In [59]:
from hybrid_search_rrf_dataset.router import (
    QueryEncoder,
    Representation,
    RouterExperiment,
    StrategyRouter,
)

# ---- config: edit and Run All ----
REPRESENTATION = Representation.ENGINEERED  # ENGINEERED | EMBEDDING | BOTH
ENCODER = None           # QueryEncoder() for EMBEDDING/BOTH (downloads e5 once)
PROTOCOL = 'random_within_lane'  # or 'holdout_lane' (train/use cells only)
ALL_ROWS = 'decisive'     # 'decisive' ~4K clear wins | 'recommended' ~12K all real winners | 'all' ~37K, tied/zero rows as negatives
MAX_CLASS_SHARE = None    # e.g. 0.5: downsample so no winner class exceeds that share

In [60]:
ROUTER_TRAIN_ROWS = "decisive"  # 'decisive' | 'recommended' | 'all' — router's fit substrate

assert _remaining(labels).empty, (
    f"{len(_remaining(labels))} queries still unscored — auto-fusion would go to the "
    "network mid-comparison; run src/scripts/autofusion_fill.py first"
)

comparison = RouterExperiment().run(
    representations=[Representation.ENGINEERED],
    all_rows=ROUTER_TRAIN_ROWS,
    autofusion=True,
)
# run() filters representations with `is`, which autoreload breaks by rebuilding the enum
assert (comparison["representation"] != "auto_fusion").any(), (
    "no router rows — Representation.ENGINEERED lost enum identity to autoreload; "
    "restart the kernel and run all"
)
display(
    comparison[
        [
            "protocol", "representation", "n_test_decisive",
            "const_dense_only", "const_pure_rrf", "const_sparse_only",
            "oracle", "router", "headroom_captured",
        ]
    ].round(4)
)

holdout_lane·engineered: 100%|██████████| 2/2 [00:00<00:00,  8.00it/s, headroom=0.153, n=206]       


,protocol,representation,n_test_decisive,const_dense_only,const_pure_rrf,const_sparse_only,oracle,router,headroom_captured
0,random_within_lane,engineered,1021,0.6310,0.2080,0.3674,0.9749,0.7124,0.2365
1,holdout_lane,engineered,206,0.6069,0.1855,0.4303,1.0000,0.6671,0.1530
2,random_within_lane,auto_fusion,1021,0.6310,0.2080,0.3674,0.9749,0.4461,-0.5377
3,holdout_lane,auto_fusion,206,0.6069,0.1855,0.4303,1.0000,0.3953,-0.5386


### 7b — How much the two disagree, and who is right when they do

Section 7 scores the two policies separately; it cannot say whether they disagree, because
two policies can reach the same mean objective by different routes. This cell puts both
route choices on the same rows and crosses them.

The router is refit here rather than reused — `run()` returns aggregates only, not per-row
routes. `_prep_train` is the same split `_run_one` uses internally, so the model is the one
section 7 scored, not a second fit with different tuning rows. Check `n` against section 7's
`n_test_decisive` for the chosen protocol; if they match, the frames are identical.

Three readouts: the agreement rate, the route-by-route crosstab (where does the disagreement
actually live), and — on the disagreement rows only — which policy matched the labelled
winner. That last one is the one that matters: disagreeing a lot is not a virtue, disagreeing
and being right is.

**Read the verdict against the substrate.** rrf wins only 4.9% of decisive rows, so any
policy that serves rrf is nearly guaranteed to be wrong *here*, and auto-fusion serves rrf
for over half of them. Expect the crosstab's `auto_fusion = pure_rrf` column to hold most of
the disagreement and most of auto-fusion's losses. That is a real cost — a hedge that pays
elsewhere does not pay on rows with a clear winner — but it is not evidence that its dense
and sparse calls are wrong. Watch also whether the router's `pure_rrf` row is empty: with
`delta = 0` and tuned thresholds it can collapse to a two-route policy, which flatters it on
this substrate for the same structural reason.

In [62]:
from hybrid_search_rrf_dataset.router import StrategyRouter, _mean_objective

DISAGREEMENT_PROTOCOL = "random_within_lane"  # or 'holdout_lane'

exp = RouterExperiment()
train, test = exp.split(DISAGREEMENT_PROTOCOL)
train_model, train_tune = exp._prep_train(train)
lr = StrategyRouter(Representation.ENGINEERED).fit(train_model, all_rows=ROUTER_TRAIN_ROWS)
lr.tune_thresholds(train_tune)

held = decisive_rows(test)
held["router"] = lr.predict_routes(held)
held["auto_fusion"] = AutoFusionRouter().route_batch(held, desc="auto-fusion")
agree = held["router"] == held["auto_fusion"]

print(f"{DISAGREEMENT_PROTOCOL}: {len(held):,} held-out decisive rows, "
      f"agree on {agree.mean():.1%}")
display(pd.crosstab(held["router"], held["auto_fusion"], margins=True))

split = held[~agree]
missed = (split["router"] != split["winner"]) & (split["auto_fusion"] != split["winner"])
display(
    pd.DataFrame(
        {
            "n": len(split),
            "router right": (split["router"] == split["winner"]).mean(),
            "auto_fusion right": (split["auto_fusion"] == split["winner"]).mean(),
            "neither": missed.mean(),
            "objective router": _mean_objective(split, list(split["router"])),
            "objective auto_fusion": _mean_objective(split, list(split["auto_fusion"])),
        },
        index=["disagreements"],
    ).round(3)
)

random_within_lane: 1,021 held-out decisive rows, agree on 46.3%


auto_fusion,dense_only,pure_rrf,sparse_only,All
router,,,,
dense_only,451,432,9,892
sparse_only,5,102,22,129
All,456,534,31,1021


,n,router right,auto_fusion right,neither,objective router,objective auto_fusion
disagreements,548,0.704,0.057,0.239,0.708,0.212


### 7c — The disagreements themselves

Five queries from each outcome bucket, not a flat sample: `auto_fusion right` is 5.7% of the
slice, so twenty random rows would show it once or not at all. `af_score` is the raw 0–9 the
classifier returned, and `obj_af` / `obj_router` are the objective each policy's chosen route
actually scored on that row — the size of the miss, not just its direction.

The crosstab above the sample is the one to read first. It splits the disagreement into three
mechanisms rather than a pile of errors, and each bucket turns out to be almost pure in the
winner it contains — which tells you the two policies are not disagreeing about *judgment*,
they are disagreeing about *whether to hedge*.

Queries keep their full text; only runs of whitespace are collapsed, because the `clerc` and
`freshstack` rows carry embedded newlines that break the table. How much of that text is
shown is `QUERY_WIDTH` in the config cell, and any single row can be printed in full —
`print(examples.loc[43, "query"])`.

In [68]:
PER_BUCKET = 5

pd.set_option("display.max_colwidth", 300)


def route_score(frame, routes):
    """Each row's objective score for the route that was actually served."""
    return _score_matrix(frame)[range(len(frame)), [_ROUTE_ORDER.index(r) for r in routes]]


examples = split.merge(read_cache(), on=["dataset", "query_id"], how="left")
examples["obj_af"] = route_score(examples, examples["auto_fusion"])
examples["obj_router"] = route_score(examples, examples["router"])
examples["outcome"] = "neither"
examples.loc[examples["router"] == examples["winner"], "outcome"] = "router right"
examples.loc[examples["auto_fusion"] == examples["winner"], "outcome"] = "auto_fusion right"
examples["query"] = (
    examples["query"].str.replace(r"\s+", " ", regex=True)
)

display(pd.crosstab(examples["outcome"], examples["winner"], margins=True))
display(
    examples.groupby("outcome")
    .sample(
        n=min(PER_BUCKET, int(examples["outcome"].value_counts().min())),
        random_state=SEED,
    )
    .sort_values("outcome")[
        [
            "outcome", "dataset", "query", "af_score",
            "auto_fusion", "router", "winner", "obj_af", "obj_router",
        ]
    ]
    .round(3)
)

winner,dense_only,pure_rrf,sparse_only,All
outcome,,,,
auto_fusion right,3,23,5,31
neither,11,0,120,131
router right,301,0,85,386
All,315,23,210,548


,outcome,dataset,query,af_score,auto_fusion,router,winner,obj_af,obj_router
43,auto_fusion right,clerc,"in demonstrative terms.” Kay-nard v. Mego Corp., supra, 484 F.Supp. at 174. The court went on to note that entrenchment of Local 807 at the Brentwood plant would do irreparable harm to Local 101’s chances of success in a representation election. We have considerably more difficulty with this por...",5,pure_rrf,sparse_only,pure_rrf,1.000,0.189
510,auto_fusion right,webfaq-eng,Who is not a candidate for PRP?,3,pure_rrf,dense_only,pure_rrf,1.000,0.189
275,auto_fusion right,msmarco-passage-dev,width of commercial egress path,3,pure_rrf,dense_only,pure_rrf,0.942,0.177
222,auto_fusion right,gooaq,how to extract data from excel using c#?,3,pure_rrf,dense_only,pure_rrf,1.000,0.150
484,auto_fusion right,scirgen-geo-en,What are the characteristics of eddy covariance systems used in hydrometeorological data collection?,3,pure_rrf,dense_only,pure_rrf,1.000,0.189
452,neither,scirgen-geo-en,What data collection methods are used for measuring vegetation spectra in China?,3,pure_rrf,dense_only,sparse_only,0.189,0.189
264,neither,lotte-technology-forum,"Im a Subversion geek, why should I consider or not consider Mercurial or Git or any other DVCS?",3,pure_rrf,dense_only,sparse_only,0.067,0.042
328,neither,orcas,pennsylvania unemployment phone number,3,pure_rrf,dense_only,sparse_only,0.143,0.183
163,neither,dbpedia-entity,What did Bruce Carver die from?,3,pure_rrf,dense_only,sparse_only,0.116,0.000
459,neither,scirgen-geo-en,Are the methods used for measuring soil respiration in different landscapes consistent with hierarchical observation techniques?,3,pure_rrf,dense_only,sparse_only,0.189,0.116
